<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/04_Candidate_Imputation_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# CELL 04.0 — LOAD CONFIGURATION AND ARTIFACTS
# ============================================================

from pathlib import Path
import json
import warnings
import gc

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04")
print("DATASET AND FEATURE PROFILING")
print("=" * 100)


# ============================================================
# 1. GOOGLE DRIVE INITIALIZATION
# ============================================================

DRIVE_MOUNT = Path("/content/drive")

print("\nGOOGLE DRIVE")
print("-" * 100)

def drive_is_ready():
    try:
        return (
            DRIVE_MOUNT.exists()
            and (DRIVE_MOUNT / "MyDrive").is_dir()
        )
    except OSError:
        return False


if not drive_is_ready():

    print("Mounting Google Drive...")

    try:
        drive.mount(
            str(DRIVE_MOUNT),
            force_remount=False
        )

    except Exception:

        print(
            "Standard mount failed. "
            "Attempting force remount..."
        )

        drive.mount(
            str(DRIVE_MOUNT),
            force_remount=True
        )

else:

    print(
        "Google Drive is already mounted."
    )


# ============================================================
# 2. PROJECT ROOT
# ============================================================

PROJECT_ROOT = (
    DRIVE_MOUNT
    / "MyDrive"
    / "AIR_LLM_Research"
)

print("\nPROJECT ROOT")
print("-" * 100)
print(f"{PROJECT_ROOT}")


try:

    PROJECT_VALID = (
        PROJECT_ROOT.exists()
        and PROJECT_ROOT.is_dir()
    )

except OSError as exc:

    raise RuntimeError(
        "Google Drive is mounted but is not currently "
        "accessible.\n\n"
        "Restart the Colab runtime, mount Drive again, "
        "and rerun this cell.\n\n"
        f"Drive error: {exc}"
    )


if not PROJECT_VALID:

    raise FileNotFoundError(
        "AIR_LLM_Research was not found at:\n\n"
        f"{PROJECT_ROOT}\n\n"
        "Expected location:\n"
        "/content/drive/MyDrive/AIR_LLM_Research"
    )

print("Status : FOUND")


# ============================================================
# 3. MAIN PROJECT DIRECTORIES
# ============================================================

DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "artifacts"
)


# ============================================================
# 4. NOTEBOOK 02 INPUTS
# ============================================================

NOTEBOOK_02_DIR = (
    DATA_DIR
    / "notebook_02"
)

SPLIT_DIR = (
    NOTEBOOK_02_DIR
    / "splits"
)

NOTEBOOK_02_METADATA_DIR = (
    NOTEBOOK_02_DIR
    / "metadata"
)

NOTEBOOK_02_PROFILE_DIR = (
    NOTEBOOK_02_DIR
    / "profiles"
)


# ============================================================
# 5. NOTEBOOK 03 INPUTS
# ============================================================

NOTEBOOK_03_DIR = (
    DATA_DIR
    / "notebook_03"
)

SCENARIO_DIR = (
    NOTEBOOK_03_DIR
    / "scenarios"
)

MASK_DIR = (
    NOTEBOOK_03_DIR
    / "masks"
)

GROUND_TRUTH_DIR = (
    NOTEBOOK_03_DIR
    / "ground_truth"
)

NOTEBOOK_03_METADATA_DIR = (
    NOTEBOOK_03_DIR
    / "metadata"
)

NOTEBOOK_03_DIAGNOSTIC_DIR = (
    NOTEBOOK_03_DIR
    / "diagnostics"
)

SCENARIO_REGISTRY_PATH = (
    NOTEBOOK_03_METADATA_DIR
    / "scenario_registry.csv"
)

NOTEBOOK_03_MANIFEST_PATH = (
    ARTIFACT_DIR
    / "notebook_03_manifest.json"
)


# ============================================================
# 6. NOTEBOOK 04 OUTPUT DIRECTORIES
# ============================================================

NOTEBOOK_04_DIR = (
    DATA_DIR
    / "notebook_04"
)

PROFILE_DIR = (
    NOTEBOOK_04_DIR
    / "profiles"
)

DATASET_PROFILE_DIR = (
    PROFILE_DIR
    / "dataset"
)

FEATURE_PROFILE_DIR = (
    PROFILE_DIR
    / "feature"
)

DEPENDENCY_DIR = (
    PROFILE_DIR
    / "dependency"
)

MISSINGNESS_PROFILE_DIR = (
    PROFILE_DIR
    / "missingness"
)

TARGET_PROFILE_DIR = (
    PROFILE_DIR
    / "target"
)

METADATA_04_DIR = (
    NOTEBOOK_04_DIR
    / "metadata"
)

DIAGNOSTIC_04_DIR = (
    NOTEBOOK_04_DIR
    / "diagnostics"
)


# ============================================================
# 7. CREATE NOTEBOOK 04 OUTPUT DIRECTORIES
# ============================================================

for directory in [

    NOTEBOOK_04_DIR,

    PROFILE_DIR,

    DATASET_PROFILE_DIR,

    FEATURE_PROFILE_DIR,

    DEPENDENCY_DIR,

    MISSINGNESS_PROFILE_DIR,

    TARGET_PROFILE_DIR,

    METADATA_04_DIR,

    DIAGNOSTIC_04_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 8. DATASET CONFIGURATION
# ============================================================

DATASETS = [

    "adult_income",

    "bank_marketing",

    "diabetes_130us"

]


TARGET_REGISTRY = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted"

}


MASTER_SEED = 42

np.random.seed(
    MASTER_SEED
)


# ============================================================
# 9. PROFILING CONFIGURATION
# ============================================================

# Maximum number of categorical levels reported
# in frequency summaries.
MAX_CATEGORY_LEVELS = 20


# Minimum observations required for categorical
# frequency analysis.
MIN_CATEGORY_OBSERVATIONS = 1


# Maximum number of features used in expensive
# pairwise dependency calculations.
MAX_DEPENDENCY_FEATURES = 100


# Maximum observations used for expensive
# dependency calculations.
DEPENDENCY_SAMPLE_SIZE = 10000


# Maximum observations used for missingness
# association analysis.
MISSINGNESS_ASSOCIATION_SAMPLE_SIZE = 10000


# Maximum observations used for mutual information.
MUTUAL_INFORMATION_SAMPLE_SIZE = 10000


# Maximum number of features used for MI analysis.
MAX_MI_FEATURES = 100


# Minimum number of non-missing observations required
# for reliable feature profiling.
MIN_PROFILE_OBSERVATIONS = 10


# ============================================================
# 10. VERIFY NOTEBOOK 02 INPUTS
# ============================================================

print("\nNOTEBOOK 02 INPUTS")
print("-" * 100)

if not SPLIT_DIR.is_dir():

    raise FileNotFoundError(
        "Notebook 02 split directory not found:\n"
        f"{SPLIT_DIR}"
    )


TRAIN_PATHS = {}

for dataset_id in DATASETS:

    train_path = (
        SPLIT_DIR
        / dataset_id
        / f"{dataset_id}_train.csv"
    )

    TRAIN_PATHS[
        dataset_id
    ] = train_path

    if not train_path.is_file():

        raise FileNotFoundError(
            f"Training file not found for "
            f"{dataset_id}:\n"
            f"{train_path}"
        )

    print(
        f"{dataset_id:20s} : FOUND"
    )


# ============================================================
# 11. VERIFY NOTEBOOK 03 ARTIFACTS
# ============================================================

print("\nNOTEBOOK 03 INPUTS")
print("-" * 100)

if not NOTEBOOK_03_DIR.is_dir():

    raise FileNotFoundError(
        "Notebook 03 directory not found:\n"
        f"{NOTEBOOK_03_DIR}"
    )


if not SCENARIO_REGISTRY_PATH.is_file():

    raise FileNotFoundError(
        "Notebook 03 scenario registry not found:\n"
        f"{SCENARIO_REGISTRY_PATH}"
    )


print(
    f"Scenario registry : FOUND"
)

print(
    f"Mask directory    : "
    f"{'FOUND' if MASK_DIR.is_dir() else 'MISSING'}"
)

print(
    f"Ground truth      : "
    f"{'FOUND' if GROUND_TRUTH_DIR.is_dir() else 'MISSING'}"
)

print(
    f"Diagnostics       : "
    f"{'FOUND' if NOTEBOOK_03_DIAGNOSTIC_DIR.is_dir() else 'MISSING'}"
)


# ============================================================
# 12. LOAD SCENARIO REGISTRY
# ============================================================

SCENARIO_REGISTRY_DF = pd.read_csv(
    SCENARIO_REGISTRY_PATH
)


if SCENARIO_REGISTRY_DF.empty:

    raise RuntimeError(
        "Notebook 03 scenario registry is empty."
    )


required_scenario_columns = [

    "scenario_id",

    "dataset_id",

    "mechanism",

    "requested_rate",

    "repetition",

    "seed"

]


missing_scenario_columns = [

    column

    for column in required_scenario_columns

    if column not in SCENARIO_REGISTRY_DF.columns

]


if missing_scenario_columns:

    raise ValueError(
        "Notebook 03 scenario registry is missing "
        "required columns:\n"
        + "\n".join(
            missing_scenario_columns
        )
    )


print("\nSCENARIO REGISTRY")
print("-" * 100)

print(
    f"Rows    : "
    f"{len(SCENARIO_REGISTRY_DF):,}"
)

print(
    f"Columns : "
    f"{len(SCENARIO_REGISTRY_DF.columns)}"
)


# ============================================================
# 13. LOAD NOTEBOOK 03 MANIFEST
# ============================================================

NOTEBOOK_03_MANIFEST = None

if NOTEBOOK_03_MANIFEST_PATH.is_file():

    try:

        with open(
            NOTEBOOK_03_MANIFEST_PATH,
            "r",
            encoding="utf-8"
        ) as file:

            NOTEBOOK_03_MANIFEST = json.load(
                file
            )

        print(
            "Notebook 03 manifest : LOADED"
        )

    except Exception as exc:

        print(
            "Notebook 03 manifest : "
            f"COULD NOT BE READ ({exc})"
        )

else:

    print(
        "Notebook 03 manifest : NOT FOUND"
    )


# ============================================================
# 14. CONFIGURATION SUMMARY
# ============================================================

print("\nPROFILING CONFIGURATION")
print("-" * 100)

print(
    f"Master seed                     : "
    f"{MASTER_SEED}"
)

print(
    f"Maximum category levels        : "
    f"{MAX_CATEGORY_LEVELS}"
)

print(
    f"Maximum dependency features    : "
    f"{MAX_DEPENDENCY_FEATURES}"
)

print(
    f"Dependency sample size         : "
    f"{DEPENDENCY_SAMPLE_SIZE:,}"
)

print(
    f"Missingness association sample : "
    f"{MISSINGNESS_ASSOCIATION_SAMPLE_SIZE:,}"
)

print(
    f"Mutual information sample      : "
    f"{MUTUAL_INFORMATION_SAMPLE_SIZE:,}"
)

print(
    f"Maximum MI features            : "
    f"{MAX_MI_FEATURES}"
)


# ============================================================
# 15. FINAL VALIDATION
# ============================================================

required_configuration = [

    "PROJECT_ROOT",

    "DATASETS",

    "TARGET_REGISTRY",

    "TRAIN_PATHS",

    "SPLIT_DIR",

    "SCENARIO_REGISTRY_DF",

    "NOTEBOOK_04_DIR",

    "PROFILE_DIR",

    "MAX_CATEGORY_LEVELS",

    "MAX_DEPENDENCY_FEATURES",

    "DEPENDENCY_SAMPLE_SIZE",

    "MISSINGNESS_ASSOCIATION_SAMPLE_SIZE",

    "MUTUAL_INFORMATION_SAMPLE_SIZE",

    "MAX_MI_FEATURES"

]


missing_configuration = [

    name

    for name in required_configuration

    if name not in globals()

]


if missing_configuration:

    raise RuntimeError(
        "Notebook 04 configuration is incomplete:\n"
        + "\n".join(
            missing_configuration
        )
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 04 CONFIGURATION READY")
print("=" * 100)

print(
    f"Project root       : {PROJECT_ROOT}"
)

print(
    f"Datasets           : {len(DATASETS)}"
)

print(
    f"Scenario records   : "
    f"{len(SCENARIO_REGISTRY_DF):,}"
)

print(
    f"Notebook 04 output : {NOTEBOOK_04_DIR}"
)

print(
    "Configuration      : VALIDATED"
)

print("=" * 100)

gc.collect()

AIR-LLM — NOTEBOOK 04
DATASET AND FEATURE PROFILING

GOOGLE DRIVE
----------------------------------------------------------------------------------------------------
Google Drive is already mounted.

PROJECT ROOT
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/AIR_LLM_Research
Status : FOUND

NOTEBOOK 02 INPUTS
----------------------------------------------------------------------------------------------------
adult_income         : FOUND
bank_marketing       : FOUND
diabetes_130us       : FOUND

NOTEBOOK 03 INPUTS
----------------------------------------------------------------------------------------------------
Scenario registry : FOUND
Mask directory    : FOUND
Ground truth      : FOUND
Diagnostics       : FOUND

SCENARIO REGISTRY
----------------------------------------------------------------------------------------------------
Rows    : 225
Columns : 7
Notebook 03 manifest : LOADED

PROFILING CONFIGURAT

686

In [11]:
# ============================================================
# CELL 04.1 — LOAD TRAINING DATA
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.1")
print("LOAD TRAINING DATA")
print("=" * 100)

import gc
import pandas as pd

# ------------------------------------------------------------
# Verify required configuration from Cell 04.0
# ------------------------------------------------------------

required_objects = [
    "DATASETS",
    "TRAIN_PATHS",
    "TARGET_REGISTRY"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Notebook 04 configuration is missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRun Cell 04.0 first."
    )


# ------------------------------------------------------------
# Load training datasets
# ------------------------------------------------------------

TRAINING_DATA = {}

TRAINING_DATA_INFO = []

for dataset_id in DATASETS:

    path = TRAIN_PATHS[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    print("\n" + "-" * 100)
    print(f"DATASET: {dataset_id}")
    print("-" * 100)
    print(f"Path   : {path}")

    # Read CSV once.
    df = pd.read_csv(
        path,
        low_memory=False
    )

    if df.empty:
        raise RuntimeError(
            f"Training dataset is empty:\n{path}"
        )

    # --------------------------------------------------------
    # Target validation
    # --------------------------------------------------------

    if target not in df.columns:
        raise ValueError(
            f"Target column '{target}' not found in "
            f"{dataset_id}.\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )

    # --------------------------------------------------------
    # Store in memory
    # --------------------------------------------------------

    TRAINING_DATA[dataset_id] = df

    TRAINING_DATA_INFO.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "target":
            target,

        "target_missing":
            int(df[target].isna().sum()),

        "total_missing_cells":
            int(df.isna().sum().sum()),

        "memory_mb":
            round(
                df.memory_usage(
                    deep=True
                ).sum() / (1024 ** 2),
                2
            )
    })

    print(
        f"Rows               : {df.shape[0]:,}"
    )

    print(
        f"Columns            : {df.shape[1]:,}"
    )

    print(
        f"Target             : {target}"
    )

    print(
        f"Missing cells      : "
        f"{int(df.isna().sum().sum()):,}"
    )

    print(
        f"Memory             : "
        f"{TRAINING_DATA_INFO[-1]['memory_mb']:.2f} MB"
    )

    print("Status             : LOADED")


# ------------------------------------------------------------
# Summary dataframe
# ------------------------------------------------------------

TRAINING_DATA_INFO_DF = pd.DataFrame(
    TRAINING_DATA_INFO
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if len(TRAINING_DATA) != len(DATASETS):

    raise RuntimeError(
        "Not all training datasets were loaded."
    )


for dataset_id in DATASETS:

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"Training data missing for: {dataset_id}"
        )


print("\n" + "=" * 100)
print("TRAINING DATA LOADING COMPLETE")
print("=" * 100)

display(
    TRAINING_DATA_INFO_DF
)

print(
    f"\nDatasets loaded : "
    f"{len(TRAINING_DATA)}"
)

print(
    "TRAINING_DATA dictionary : READY"
)

gc.collect()

AIR-LLM — NOTEBOOK 04.1
LOAD TRAINING DATA

----------------------------------------------------------------------------------------------------
DATASET: adult_income
----------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_02/splits/adult_income/adult_income_train.csv
Rows               : 22,792
Columns            : 16
Target             : income
Missing cells      : 2,963
Memory             : 12.48 MB
Status             : LOADED

----------------------------------------------------------------------------------------------------
DATASET: bank_marketing
----------------------------------------------------------------------------------------------------
Path   : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_02/splits/bank_marketing/bank_marketing_train.csv
Rows               : 31,647
Columns            : 18
Target             : y
Missing cells      : 0
Memory             :

,dataset_id,rows,columns,target,target_missing,total_missing_cells,memory_mb
0,adult_income,22792,16,income,0,2963,12.48
1,bank_marketing,31647,18,y,0,0,18.26
2,diabetes_130us,71236,49,readmitted,0,261834,132.15



Datasets loaded : 3
TRAINING_DATA dictionary : READY


31

In [12]:
# ============================================================
# NOTEBOOK 04.2 — LOAD NOTEBOOK 03 SCENARIO REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.2")
print("NOTEBOOK 03 SCENARIO REGISTRY VALIDATION")
print("=" * 100)


required_registry_columns = [
    "scenario_id",
    "dataset_id",
    "mechanism",
    "requested_rate",
    "repetition",
    "seed"
]

missing_columns = [
    c for c in required_registry_columns
    if c not in SCENARIO_REGISTRY_DF.columns
]

if missing_columns:
    raise ValueError(
        "Scenario registry is missing required columns:\n"
        + "\n".join(missing_columns)
    )


registry_datasets = set(
    SCENARIO_REGISTRY_DF[
        "dataset_id"
    ].astype(str)
)

if registry_datasets != set(DATASETS):
    raise ValueError(
        "Scenario registry datasets do not match "
        "Notebook 04 datasets."
    )


print(f"Scenario rows : {len(SCENARIO_REGISTRY_DF):,}")
print(
    f"Datasets      : "
    f"{SCENARIO_REGISTRY_DF['dataset_id'].nunique()}"
)

print(
    f"Mechanisms    : "
    f"{SCENARIO_REGISTRY_DF['mechanism'].nunique()}"
)

print(
    f"Rates         : "
    f"{SCENARIO_REGISTRY_DF['requested_rate'].nunique()}"
)

print(
    f"Repetitions   : "
    f"{SCENARIO_REGISTRY_DF['repetition'].nunique()}"
)

print("\nScenario registry validation: PASSED")

AIR-LLM — NOTEBOOK 04.2
NOTEBOOK 03 SCENARIO REGISTRY VALIDATION
Scenario rows : 225
Datasets      : 3
Mechanisms    : 3
Rates         : 5
Repetitions   : 5

Scenario registry validation: PASSED


In [13]:
# ============================================================
# NOTEBOOK 04.3 — DATASET-LEVEL PROFILE
# ============================================================

DATASET_PROFILE_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.3")
print("DATASET-LEVEL PROFILING")
print("=" * 100)


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    feature_columns = [
        c for c in df.columns
        if c != target
    ]

    numeric_features = [
        c for c in feature_columns
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    categorical_features = [
        c for c in feature_columns
        if c not in numeric_features
    ]

    total_cells = (
        len(df)
        * len(feature_columns)
    )

    missing_cells = int(
        df[feature_columns]
        .isna()
        .sum()
        .sum()
    )

    missing_rate = (
        missing_cells / total_cells
        if total_cells > 0
        else 0.0
    )

    incomplete_features = int(
        df[feature_columns]
        .isna()
        .any()
        .sum()
    )

    duplicate_rows = int(
        df.duplicated().sum()
    )

    duplicate_ratio = (
        duplicate_rows / len(df)
        if len(df) > 0
        else 0.0
    )

    class_count = int(
        df[target].nunique(
            dropna=True
        )
    )

    target_missing = int(
        df[target].isna().sum()
    )

    DATASET_PROFILE_ROWS.append({

        "dataset_id": dataset_id,

        "row_count":
            int(len(df)),

        "column_count":
            int(len(df.columns)),

        "feature_count":
            int(len(feature_columns)),

        "numeric_feature_count":
            int(len(numeric_features)),

        "categorical_feature_count":
            int(len(categorical_features)),

        "numeric_ratio":
            len(numeric_features)
            / max(len(feature_columns), 1),

        "categorical_ratio":
            len(categorical_features)
            / max(len(feature_columns), 1),

        "total_cells":
            int(total_cells),

        "missing_cells":
            missing_cells,

        "overall_missing_rate":
            missing_rate,

        "incomplete_feature_count":
            incomplete_features,

        "duplicate_rows":
            duplicate_rows,

        "duplicate_ratio":
            duplicate_ratio,

        "target":
            target,

        "target_unique_values":
            class_count,

        "target_missing_count":
            target_missing,

        "target_missing_rate":
            target_missing
            / max(len(df), 1)

    })


DATASET_PROFILE_DF = pd.DataFrame(
    DATASET_PROFILE_ROWS
)

display(
    DATASET_PROFILE_DF
)

AIR-LLM — NOTEBOOK 04.3
DATASET-LEVEL PROFILING


,dataset_id,row_count,column_count,feature_count,numeric_feature_count,categorical_feature_count,numeric_ratio,categorical_ratio,total_cells,missing_cells,overall_missing_rate,incomplete_feature_count,duplicate_rows,duplicate_ratio,target,target_unique_values,target_missing_count,target_missing_rate
0,adult_income,22792,16,15,7,8,0.466667,0.533333,341880,2963,0.008667,3,0,0.0,income,2,0,0.0
1,bank_marketing,31647,18,17,8,9,0.470588,0.529412,537999,0,0.000000,0,0,0.0,y,2,0,0.0
2,diabetes_130us,71236,49,48,12,36,0.250000,0.750000,3419328,261834,0.076575,9,0,0.0,readmitted,3,0,0.0


In [14]:
# ============================================================
# NOTEBOOK 04.4 — NUMERICAL FEATURE PROFILE
# ============================================================

NUMERICAL_PROFILE_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.4")
print("NUMERICAL FEATURE PROFILING")
print("=" * 100)


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    for feature in df.columns:

        if feature == target:
            continue

        series = df[feature]

        if not pd.api.types.is_numeric_dtype(series):
            continue

        observed = series.dropna()

        if len(observed) == 0:
            continue

        q1 = float(
            observed.quantile(0.25)
        )

        q3 = float(
            observed.quantile(0.75)
        )

        iqr = q3 - q1

        if iqr > 0:

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            outlier_count = int(
                (
                    (observed < lower)
                    |
                    (observed > upper)
                ).sum()
            )

        else:
            outlier_count = 0

        NUMERICAL_PROFILE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "dtype":
                str(series.dtype),

            "observed_count":
                int(observed.size),

            "missing_count":
                int(series.isna().sum()),

            "missing_rate":
                float(series.isna().mean()),

            "unique_count":
                int(observed.nunique()),

            "mean":
                float(observed.mean()),

            "std":
                float(observed.std()),

            "variance":
                float(observed.var()),

            "min":
                float(observed.min()),

            "q1":
                q1,

            "median":
                float(observed.median()),

            "q3":
                q3,

            "max":
                float(observed.max()),

            "skewness":
                float(observed.skew()),

            "outlier_count":
                outlier_count,

            "outlier_rate":
                outlier_count
                / max(len(observed), 1)

        })


NUMERICAL_PROFILE_DF = pd.DataFrame(
    NUMERICAL_PROFILE_ROWS
)

print(
    f"Numerical features profiled: "
    f"{len(NUMERICAL_PROFILE_DF):,}"
)

display(
    NUMERICAL_PROFILE_DF.head(20)
)

gc.collect()

AIR-LLM — NOTEBOOK 04.4
NUMERICAL FEATURE PROFILING
Numerical features profiled: 27


,dataset_id,feature,dtype,observed_count,missing_count,missing_rate,unique_count,mean,std,variance,min,q1,median,q3,max,skewness,outlier_count,outlier_rate
0,adult_income,__air_llm_row_id,int64,22792,0,0.0,22792,16220.811995,9376.973867,8.792764e+07,0.0,8122.75,16186.5,24319.25,32560.0,0.007883,0,0.000000
1,adult_income,age,int64,22792,0,0.0,71,38.583538,13.631512,1.858181e+02,17.0,28.00,37.0,47.00,90.0,0.564833,165,0.007239
2,adult_income,fnlwgt,int64,22792,0,0.0,16719,190173.628993,105865.625771,1.120753e+10,14878.0,118133.75,178854.5,237544.00,1484705.0,1.499653,689,0.030230
3,adult_income,education_num,int64,22792,0,0.0,16,10.074675,2.589412,6.705055e+00,1.0,9.00,10.0,12.00,16.0,-0.327423,875,0.038391
4,adult_income,capital_gain,int64,22792,0,0.0,113,1064.314847,7287.642808,5.310974e+07,0.0,0.00,0.0,0.00,99999.0,12.079008,0,0.000000
5,adult_income,capital_loss,int64,22792,0,0.0,86,86.372762,401.452577,1.611642e+05,0.0,0.00,0.0,0.00,4356.0,4.654249,0,0.000000
6,adult_income,hours_per_week,int64,22792,0,0.0,93,40.469112,12.417125,1.541850e+02,1.0,40.00,40.0,45.00,99.0,0.255465,6269,0.275053
7,bank_marketing,__air_llm_row_id,int64,31647,0,0.0,31647,22602.388062,13044.705146,1.701643e+08,0.0,11303.50,22595.0,33911.50,45210.0,-0.001197,0,0.000000
8,bank_marketing,age,int64,31647,0,0.0,77,40.883022,10.621809,1.128228e+02,18.0,33.00,39.0,48.00,95.0,0.697187,344,0.010870
9,bank_marketing,balance,int64,31647,0,0.0,6255,1363.589535,3070.235315,9.426345e+06,-8019.0,74.00,451.0,1427.50,102127.0,8.481049,3318,0.104844


31

In [15]:
# ============================================================
# NOTEBOOK 04.5 — CATEGORICAL FEATURE PROFILE
# ============================================================

CATEGORICAL_PROFILE_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.5")
print("CATEGORICAL FEATURE PROFILING")
print("=" * 100)


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    for feature in df.columns:

        if feature == target:
            continue

        series = df[feature]

        if pd.api.types.is_numeric_dtype(series):
            continue

        observed = series.dropna()

        if len(observed) == 0:
            continue

        value_counts = (
            observed
            .value_counts(
                normalize=True,
                dropna=False
            )
        )

        top_frequency = float(
            value_counts.iloc[0]
        )

        entropy = float(
            -np.sum(
                value_counts.values
                *
                np.log2(
                    np.maximum(
                        value_counts.values,
                        1e-12
                    )
                )
            )
        )

        top_values = (
            observed
            .value_counts()
            .head(MAX_CATEGORY_LEVELS)
        )

        top_levels = "|".join(
            str(x)
            for x in top_values.index
        )

        CATEGORICAL_PROFILE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "dtype":
                str(series.dtype),

            "observed_count":
                int(len(observed)),

            "missing_count":
                int(series.isna().sum()),

            "missing_rate":
                float(series.isna().mean()),

            "unique_count":
                int(observed.nunique()),

            "top_category_frequency":
                top_frequency,

            "entropy":
                entropy,

            "top_categories":
                top_levels

        })


CATEGORICAL_PROFILE_DF = pd.DataFrame(
    CATEGORICAL_PROFILE_ROWS
)

print(
    f"Categorical features profiled: "
    f"{len(CATEGORICAL_PROFILE_DF):,}"
)

display(
    CATEGORICAL_PROFILE_DF.head(20)
)

gc.collect()

AIR-LLM — NOTEBOOK 04.5
CATEGORICAL FEATURE PROFILING
Categorical features profiled: 53


,dataset_id,feature,dtype,observed_count,missing_count,missing_rate,unique_count,top_category_frequency,entropy,top_categories
0,adult_income,workclass,object,21517,1275,0.055941,8,0.735976,1.426176,Private|Self-emp-not-inc|Local-gov|State-gov|S...
1,adult_income,education,object,22792,0,0.000000,16,0.319717,2.946977,HS-grad|Some-college|Bachelors|Masters|Assoc-v...
2,adult_income,marital_status,object,22792,0,0.000000,7,0.458406,1.842286,Married-civ-spouse|Never-married|Divorced|Sepa...
3,adult_income,occupation,object,21511,1281,0.056204,14,0.136628,3.397831,Prof-specialty|Craft-repair|Exec-managerial|Ad...
4,adult_income,relationship,object,22792,0,0.000000,6,0.404133,2.155002,Husband|Not-in-family|Own-child|Unmarried|Wife...
5,adult_income,race,object,22792,0,0.000000,5,0.855125,0.796937,White|Black|Asian-Pac-Islander|Amer-Indian-Esk...
6,adult_income,sex,object,22792,0,0.000000,2,0.667778,0.917180,Male|Female
7,adult_income,native_country,object,22385,407,0.017857,41,0.911548,0.829732,United-States|Mexico|Philippines|Germany|Canad...
8,bank_marketing,job,object,31647,0,0.000000,12,0.216956,3.059065,blue-collar|management|technician|admin.|servi...
9,bank_marketing,marital,object,31647,0,0.000000,3,0.601921,1.314757,married|single|divorced


31

In [16]:
# ============================================================
# NOTEBOOK 04.6 — MISSINGNESS FEATURE PROFILE
# ============================================================

MISSINGNESS_PROFILE_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.6")
print("MISSINGNESS FEATURE PROFILING")
print("=" * 100)


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    feature_columns = [
        c for c in df.columns
        if c != target
    ]

    for feature in feature_columns:

        missing_indicator = (
            df[feature].isna()
        )

        missing_count = int(
            missing_indicator.sum()
        )

        missing_rate = float(
            missing_indicator.mean()
        )

        observed_count = (
            len(df)
            - missing_count
        )

        row_missing_counts = (
            df[feature_columns]
            .isna()
            .sum(axis=1)
        )

        if missing_count > 0:

            missing_row_mean = float(
                row_missing_counts[
                    missing_indicator
                ].mean()
            )

            observed_row_mean = float(
                row_missing_counts[
                    ~missing_indicator
                ].mean()
            )

        else:

            missing_row_mean = 0.0
            observed_row_mean = float(
                row_missing_counts.mean()
            )

        MISSINGNESS_PROFILE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "missing_count":
                missing_count,

            "observed_count":
                observed_count,

            "missing_percentage":
                missing_rate * 100,

            "missing_rate":
                missing_rate,

            "missing_indicator_mean":
                missing_rate,

            "mean_row_missing_features_when_missing":
                missing_row_mean,

            "mean_row_missing_features_when_observed":
                observed_row_mean,

            "missingness_burden":
                missing_rate
                * len(df)

        })


MISSINGNESS_PROFILE_DF = pd.DataFrame(
    MISSINGNESS_PROFILE_ROWS
)

display(
    MISSINGNESS_PROFILE_DF.head(20)
)

gc.collect()

AIR-LLM — NOTEBOOK 04.6
MISSINGNESS FEATURE PROFILING


,dataset_id,feature,missing_count,observed_count,missing_percentage,missing_rate,missing_indicator_mean,mean_row_missing_features_when_missing,mean_row_missing_features_when_observed,missingness_burden
0,adult_income,__air_llm_row_id,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
1,adult_income,age,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
2,adult_income,workclass,1275,21517,5.594068,0.055941,0.055941,2.015686,0.018265,1275.0
3,adult_income,fnlwgt,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
4,adult_income,education,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
5,adult_income,education_num,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
6,adult_income,marital_status,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
7,adult_income,occupation,1281,21511,5.620393,0.056204,0.056204,2.010929,0.017991,1281.0
8,adult_income,relationship,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0
9,adult_income,race,0,22792,0.000000,0.000000,0.000000,0.000000,0.130002,0.0


31

In [17]:
# ============================================================
# NOTEBOOK 04.7 — DEPENDENCY ANALYSIS
# ============================================================

NUMERIC_DEPENDENCY_ROWS = []
CATEGORICAL_ASSOCIATION_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.7")
print("DEPENDENCY ANALYSIS")
print("=" * 100)


def cramers_v(x, y):

    table = pd.crosstab(
        x,
        y
    )

    if table.empty:
        return np.nan

    observed = table.to_numpy(
        dtype=float
    )

    total = observed.sum()

    if total == 0:
        return np.nan

    row_sum = observed.sum(axis=1)
    col_sum = observed.sum(axis=0)

    expected = (
        np.outer(
            row_sum,
            col_sum
        )
        / total
    )

    mask = expected > 0

    chi2 = (
        (
            (observed - expected) ** 2
            / np.where(
                mask,
                expected,
                1
            )
        )[mask]
    ).sum()

    n = observed.sum()

    phi2 = chi2 / max(n, 1)

    r, k = observed.shape

    if n <= 1:
        return np.nan

    phi2corr = max(
        0,
        phi2
        - ((k - 1) * (r - 1))
        / (n - 1)
    )

    rcorr = (
        r
        - ((r - 1) ** 2)
        / (n - 1)
    )

    kcorr = (
        k
        - ((k - 1) ** 2)
        / (n - 1)
    )

    denominator = min(
        kcorr - 1,
        rcorr - 1
    )

    if denominator <= 0:
        return 0.0

    return float(
        np.sqrt(
            phi2corr
            / denominator
        )
    )


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    feature_columns = [
        c for c in df.columns
        if c != target
    ]

    numeric_features = [
        c for c in feature_columns
        if pd.api.types.is_numeric_dtype(
            df[c]
        )
    ]

    categorical_features = [
        c for c in feature_columns
        if c not in numeric_features
    ]


    # --------------------------------------------------------
    # Numeric correlations
    # --------------------------------------------------------

    if numeric_features:

        corr_df = df[
            numeric_features
        ].corr(
            method="spearman"
        )

        for i, feature_a in enumerate(
            numeric_features
        ):

            for feature_b in numeric_features[
                i + 1:
            ]:

                value = corr_df.loc[
                    feature_a,
                    feature_b
                ]

                NUMERIC_DEPENDENCY_ROWS.append({

                    "dataset_id":
                        dataset_id,

                    "feature_a":
                        feature_a,

                    "feature_b":
                        feature_b,

                    "dependency_type":
                        "spearman",

                    "dependency_value":
                        float(value)
                        if pd.notna(value)
                        else np.nan,

                    "absolute_dependency":
                        abs(float(value))
                        if pd.notna(value)
                        else np.nan

                })


    # --------------------------------------------------------
    # Categorical association
    # --------------------------------------------------------

    # Limit expensive pairwise analysis.
    categorical_for_assoc = (
        categorical_features[:30]
    )

    for i, feature_a in enumerate(
        categorical_for_assoc
    ):

        for feature_b in categorical_for_assoc[
            i + 1:
        ]:

            pair = df[
                [feature_a, feature_b]
            ].dropna()

            if len(pair) == 0:
                continue

            # Prevent extremely high-cardinality
            # contingency tables.
            if (
                pair[feature_a].nunique()
                > 100
                or
                pair[feature_b].nunique()
                > 100
            ):
                continue

            value = cramers_v(
                pair[feature_a],
                pair[feature_b]
            )

            CATEGORICAL_ASSOCIATION_ROWS.append({

                "dataset_id":
                    dataset_id,

                "feature_a":
                    feature_a,

                "feature_b":
                    feature_b,

                "dependency_type":
                    "cramers_v",

                "dependency_value":
                    value,

                "absolute_dependency":
                    abs(value)
                    if pd.notna(value)
                    else np.nan

            })


NUMERIC_DEPENDENCY_DF = pd.DataFrame(
    NUMERIC_DEPENDENCY_ROWS
)

CATEGORICAL_ASSOCIATION_DF = pd.DataFrame(
    CATEGORICAL_ASSOCIATION_ROWS
)


print(
    f"Numeric dependency pairs      : "
    f"{len(NUMERIC_DEPENDENCY_DF):,}"
)

print(
    f"Categorical association pairs : "
    f"{len(CATEGORICAL_ASSOCIATION_DF):,}"
)


gc.collect()

AIR-LLM — NOTEBOOK 04.7
DEPENDENCY ANALYSIS
Numeric dependency pairs      : 115
Categorical association pairs : 414


0

In [19]:
# ============================================================
# NOTEBOOK 04.8 — MUTUAL INFORMATION ANALYSIS
# ============================================================

import gc
import numpy as np
import pandas as pd

from sklearn.feature_selection import (
    mutual_info_regression,
    mutual_info_classif
)

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.8")
print("MUTUAL INFORMATION ANALYSIS")
print("=" * 100)


# ============================================================
# 1. PROFILING CONFIGURATION
# ============================================================

# Use existing global configuration when available.
# Otherwise, define a reproducible default.

if "PROFILE_SAMPLE_SIZE" not in globals():
    PROFILE_SAMPLE_SIZE = 10_000

if "MASTER_SEED" not in globals():
    MASTER_SEED = 42


print(f"Profiling sample size : {PROFILE_SAMPLE_SIZE:,}")
print(f"Random seed           : {MASTER_SEED}")


# ============================================================
# 2. VALIDATE REQUIRED OBJECTS
# ============================================================

required_objects = [
    "DATASETS",
    "TRAINING_DATA",
    "TARGET_REGISTRY"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        "Required AIR-LLM objects are missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 3. INITIALIZE RESULTS
# ============================================================

MI_ROWS = []


# ============================================================
# 4. HELPER FUNCTION — SAFE NUMERIC CONVERSION
# ============================================================

def prepare_numeric_series(series):
    """
    Convert a feature to numeric values while preserving
    missing observations for later median imputation.
    """

    numeric_series = pd.to_numeric(
        series,
        errors="coerce"
    )

    if numeric_series.notna().sum() == 0:
        return None

    median_value = numeric_series.median()

    if pd.isna(median_value):
        return None

    return numeric_series.fillna(median_value)


# ============================================================
# 5. HELPER FUNCTION — SAFE CATEGORICAL ENCODING
# ============================================================

def prepare_categorical_series(series):
    """
    Convert categorical values into integer codes.
    Missing values are represented explicitly.
    """

    values = (
        series
        .fillna("__MISSING__")
        .astype(str)
    )

    codes, uniques = pd.factorize(
        values,
        sort=True
    )

    if len(uniques) <= 1:
        return None

    return codes


# ============================================================
# 6. MAIN MUTUAL INFORMATION ANALYSIS
# ============================================================

for dataset_id in DATASETS:

    print("\n" + "-" * 100)
    print(f"DATASET: {dataset_id}")
    print("-" * 100)


    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------

    if dataset_id not in TRAINING_DATA:
        print(
            f"WARNING: Training data unavailable for "
            f"{dataset_id}. Skipping."
        )
        continue

    df = TRAINING_DATA[dataset_id]


    # --------------------------------------------------------
    # Validate target
    # --------------------------------------------------------

    if dataset_id not in TARGET_REGISTRY:
        print(
            f"WARNING: Target not registered for "
            f"{dataset_id}. Skipping."
        )
        continue

    target = TARGET_REGISTRY[dataset_id]


    if target not in df.columns:
        print(
            f"WARNING: Target '{target}' not found in "
            f"{dataset_id}. Skipping."
        )
        continue


    # --------------------------------------------------------
    # Feature columns
    # --------------------------------------------------------

    feature_columns = [
        c
        for c in df.columns
        if c != target
    ]


    if len(feature_columns) == 0:
        print(
            f"WARNING: No features available for "
            f"{dataset_id}. Skipping."
        )
        continue


    # --------------------------------------------------------
    # Profiling sample
    # --------------------------------------------------------

    if len(df) > PROFILE_SAMPLE_SIZE:

        sample_df = df.sample(
            n=PROFILE_SAMPLE_SIZE,
            random_state=MASTER_SEED
        )

    else:

        sample_df = df.copy()


    print(
        f"Rows analyzed: {len(sample_df):,}"
    )


    # ========================================================
    # TARGET TYPE DETECTION
    # ========================================================

    target_is_numeric = pd.api.types.is_numeric_dtype(
        sample_df[target]
    )

    if target_is_numeric:

        target_type = "numeric"

    else:

        target_type = "categorical"


    print(
        f"Target       : {target}"
    )

    print(
        f"Target type  : {target_type}"
    )


    # ========================================================
    # FEATURE TYPE DETECTION
    # ========================================================

    numeric_features = [
        c
        for c in feature_columns
        if pd.api.types.is_numeric_dtype(
            sample_df[c]
        )
    ]

    categorical_features = [
        c
        for c in feature_columns
        if c not in numeric_features
    ]


    print(
        f"Numeric features     : "
        f"{len(numeric_features):,}"
    )

    print(
        f"Categorical features : "
        f"{len(categorical_features):,}"
    )


    # ========================================================
    # CASE 1 — NUMERICAL FEATURE → NUMERICAL TARGET
    # ========================================================

    if target_type == "numeric":

        y = prepare_numeric_series(
            sample_df[target]
        )

        if y is not None:

            for feature in numeric_features:

                x = prepare_numeric_series(
                    sample_df[feature]
                )

                if x is None:
                    mi = np.nan

                elif x.nunique() <= 1:
                    mi = np.nan

                else:

                    try:

                        mi = mutual_info_regression(
                            x.to_numpy().reshape(-1, 1),
                            y.to_numpy(),
                            random_state=MASTER_SEED
                        )[0]

                    except Exception as e:

                        print(
                            f"MI failed for "
                            f"{dataset_id} | "
                            f"{feature}: {e}"
                        )

                        mi = np.nan


                MI_ROWS.append({

                    "dataset_id":
                        dataset_id,

                    "feature":
                        feature,

                    "target":
                        target,

                    "feature_type":
                        "numeric",

                    "target_type":
                        "numeric",

                    "mutual_information":
                        (
                            float(mi)
                            if pd.notna(mi)
                            else np.nan
                        )

                })


    # ========================================================
    # CASE 2 — CATEGORICAL FEATURE → NUMERICAL TARGET
    # ========================================================

    if target_type == "numeric":

        y = prepare_numeric_series(
            sample_df[target]
        )

        if y is not None:

            for feature in categorical_features:

                x_codes = prepare_categorical_series(
                    sample_df[feature]
                )

                if x_codes is None:
                    mi = np.nan

                elif len(np.unique(x_codes)) <= 1:
                    mi = np.nan

                else:

                    try:

                        mi = mutual_info_regression(
                            x_codes.reshape(-1, 1),
                            y.to_numpy(),
                            discrete_features=True,
                            random_state=MASTER_SEED
                        )[0]

                    except Exception as e:

                        print(
                            f"MI failed for "
                            f"{dataset_id} | "
                            f"{feature}: {e}"
                        )

                        mi = np.nan


                MI_ROWS.append({

                    "dataset_id":
                        dataset_id,

                    "feature":
                        feature,

                    "target":
                        target,

                    "feature_type":
                        "categorical",

                    "target_type":
                        "numeric",

                    "mutual_information":
                        (
                            float(mi)
                            if pd.notna(mi)
                            else np.nan
                        )

                })


    # ========================================================
    # CASE 3 — NUMERICAL FEATURE → CATEGORICAL TARGET
    # ========================================================

    if target_type == "categorical":

        y_codes = prepare_categorical_series(
            sample_df[target]
        )

        if y_codes is not None:

            for feature in numeric_features:

                x = prepare_numeric_series(
                    sample_df[feature]
                )

                if x is None:

                    mi = np.nan

                elif x.nunique() <= 1:

                    mi = np.nan

                else:

                    try:

                        mi = mutual_info_classif(
                            x.to_numpy().reshape(-1, 1),
                            y_codes,
                            discrete_features=False,
                            random_state=MASTER_SEED
                        )[0]

                    except Exception as e:

                        print(
                            f"MI failed for "
                            f"{dataset_id} | "
                            f"{feature}: {e}"
                        )

                        mi = np.nan


                MI_ROWS.append({

                    "dataset_id":
                        dataset_id,

                    "feature":
                        feature,

                    "target":
                        target,

                    "feature_type":
                        "numeric",

                    "target_type":
                        "categorical",

                    "mutual_information":
                        (
                            float(mi)
                            if pd.notna(mi)
                            else np.nan
                        )

                })


    # ========================================================
    # CASE 4 — CATEGORICAL FEATURE → CATEGORICAL TARGET
    # ========================================================

    if target_type == "categorical":

        y_codes = prepare_categorical_series(
            sample_df[target]
        )

        if y_codes is not None:

            for feature in categorical_features:

                x_codes = prepare_categorical_series(
                    sample_df[feature]
                )

                if x_codes is None:

                    mi = np.nan

                elif len(np.unique(x_codes)) <= 1:

                    mi = np.nan

                else:

                    try:

                        mi = mutual_info_classif(
                            x_codes.reshape(-1, 1),
                            y_codes,
                            discrete_features=True,
                            random_state=MASTER_SEED
                        )[0]

                    except Exception as e:

                        print(
                            f"MI failed for "
                            f"{dataset_id} | "
                            f"{feature}: {e}"
                        )

                        mi = np.nan


                MI_ROWS.append({

                    "dataset_id":
                        dataset_id,

                    "feature":
                        feature,

                    "target":
                        target,

                    "feature_type":
                        "categorical",

                    "target_type":
                        "categorical",

                    "mutual_information":
                        (
                            float(mi)
                            if pd.notna(mi)
                            else np.nan
                        )

                })


# ============================================================
# 7. CREATE MUTUAL INFORMATION DATAFRAME
# ============================================================

MUTUAL_INFORMATION_DF = pd.DataFrame(
    MI_ROWS
)


# ============================================================
# 8. VALIDATE RESULTS
# ============================================================

if MUTUAL_INFORMATION_DF.empty:

    print("\nWARNING: No mutual-information records generated.")

else:

    # Ensure numerical consistency
    MUTUAL_INFORMATION_DF[
        "mutual_information"
    ] = pd.to_numeric(
        MUTUAL_INFORMATION_DF[
            "mutual_information"
        ],
        errors="coerce"
    )


    # --------------------------------------------------------
    # Remove invalid negative MI values
    # --------------------------------------------------------

    MUTUAL_INFORMATION_DF.loc[
        MUTUAL_INFORMATION_DF[
            "mutual_information"
        ] < 0,
        "mutual_information"
    ] = np.nan


    # --------------------------------------------------------
    # Rank MI within each dataset
    # --------------------------------------------------------

    MUTUAL_INFORMATION_DF[
        "mi_rank"
    ] = (
        MUTUAL_INFORMATION_DF
        .groupby("dataset_id")[
            "mutual_information"
        ]
        .rank(
            method="min",
            ascending=False
        )
    )


# ============================================================
# 9. SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("MUTUAL INFORMATION ANALYSIS — SUMMARY")
print("=" * 100)

print(
    f"Total MI records : "
    f"{len(MUTUAL_INFORMATION_DF):,}"
)

if not MUTUAL_INFORMATION_DF.empty:

    valid_mi = (
        MUTUAL_INFORMATION_DF[
            "mutual_information"
        ]
        .notna()
        .sum()
    )

    print(
        f"Valid MI values  : "
        f"{valid_mi:,}"
    )

    print(
        f"Missing MI values: "
        f"{len(MUTUAL_INFORMATION_DF) - valid_mi:,}"
    )

    print(
        "\nRecords by dataset:"
    )

    print(
        MUTUAL_INFORMATION_DF[
            "dataset_id"
        ]
        .value_counts()
        .sort_index()
    )


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

if not MUTUAL_INFORMATION_DF.empty:

    display(
        MUTUAL_INFORMATION_DF
        .sort_values(
            [
                "dataset_id",
                "mutual_information"
            ],
            ascending=[
                True,
                False
            ],
            na_position="last"
        )
        .head(30)
    )


# ============================================================
# 11. MEMORY CLEANUP
# ============================================================

gc.collect()

print("\n" + "=" * 100)
print("NOTEBOOK 04.8 — MUTUAL INFORMATION ANALYSIS COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 04.8
MUTUAL INFORMATION ANALYSIS
Profiling sample size : 10,000
Random seed           : 42

----------------------------------------------------------------------------------------------------
DATASET: adult_income
----------------------------------------------------------------------------------------------------
Rows analyzed: 10,000
Target       : income
Target type  : categorical
Numeric features     : 7
Categorical features : 8

----------------------------------------------------------------------------------------------------
DATASET: bank_marketing
----------------------------------------------------------------------------------------------------
Rows analyzed: 10,000
Target       : y
Target type  : categorical
Numeric features     : 8
Categorical features : 9

----------------------------------------------------------------------------------------------------
DATASET: diabetes_130us
---------------------------------------------------------------------------

,dataset_id,feature,target,feature_type,target_type,mutual_information,mi_rank
11,adult_income,relationship,income,categorical,categorical,0.112793,1.0
9,adult_income,marital_status,income,categorical,categorical,0.104163,2.0
4,adult_income,capital_gain,income,numeric,categorical,0.083629,3.0
3,adult_income,education_num,income,numeric,categorical,0.071534,4.0
8,adult_income,education,income,categorical,categorical,0.065856,5.0
10,adult_income,occupation,income,categorical,categorical,0.061990,6.0
1,adult_income,age,income,numeric,categorical,0.057178,7.0
5,adult_income,capital_loss,income,numeric,categorical,0.036198,8.0
6,adult_income,hours_per_week,income,numeric,categorical,0.035510,9.0
13,adult_income,sex,income,categorical,categorical,0.024467,10.0



NOTEBOOK 04.8 — MUTUAL INFORMATION ANALYSIS COMPLETE


In [20]:
# ============================================================
# NOTEBOOK 04.9 — TARGET / TASK RELEVANCE
# ============================================================

TASK_RELEVANCE_ROWS = []

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.9")
print("TARGET / TASK RELEVANCE")
print("=" * 100)


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    feature_columns = [
        c for c in df.columns
        if c != target
    ]

    target_type = (
        "numeric"
        if pd.api.types.is_numeric_dtype(
            df[target]
        )
        else "categorical"
    )

    target_missing_rate = float(
        df[target].isna().mean()
    )

    target_cardinality = int(
        df[target].nunique(
            dropna=True
        )
    )

    for feature in feature_columns:

        row = {

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "target":
                target,

            "target_type":
                target_type,

            "target_cardinality":
                target_cardinality,

            "target_missing_rate":
                target_missing_rate

        }

        # Numerical -> numerical/categorical
        if pd.api.types.is_numeric_dtype(
            df[feature]
        ):

            pair = df[
                [feature, target]
            ].dropna()

            if len(pair) >= 3:

                if target_type == "numeric":

                    pearson = pair[
                        feature
                    ].corr(
                        pair[target],
                        method="pearson"
                    )

                    spearman = pair[
                        feature
                    ].corr(
                        pair[target],
                        method="spearman"
                    )

                else:

                    encoded_target = (
                        pd.factorize(
                            pair[target].astype(str)
                        )[0]
                    )

                    pearson = np.corrcoef(
                        pair[feature].to_numpy(),
                        encoded_target
                    )[0, 1]

                    spearman = np.corrcoef(
                        pair[feature].rank(),
                        pd.Series(
                            encoded_target
                        ).rank()
                    )[0, 1]

            else:

                pearson = np.nan
                spearman = np.nan

            row.update({

                "target_relevance_type":
                    "correlation",

                "target_relevance_primary":
                    float(pearson)
                    if pd.notna(pearson)
                    else np.nan,

                "target_relevance_secondary":
                    float(spearman)
                    if pd.notna(spearman)
                    else np.nan

            })

        else:

            pair = df[
                [feature, target]
            ].dropna()

            if len(pair) > 0:

                association = cramers_v(
                    pair[feature].astype(str),
                    pair[target].astype(str)
                )

            else:

                association = np.nan

            row.update({

                "target_relevance_type":
                    "cramers_v",

                "target_relevance_primary":
                    association,

                "target_relevance_secondary":
                    np.nan

            })

        TASK_RELEVANCE_ROWS.append(row)


TASK_RELEVANCE_DF = pd.DataFrame(
    TASK_RELEVANCE_ROWS
)

display(
    TASK_RELEVANCE_DF.head(20)
)

gc.collect()

AIR-LLM — NOTEBOOK 04.9
TARGET / TASK RELEVANCE


,dataset_id,feature,target,target_type,target_cardinality,target_missing_rate,target_relevance_type,target_relevance_primary,target_relevance_secondary
0,adult_income,__air_llm_row_id,income,categorical,2,0.0,correlation,0.003942,0.003909
1,adult_income,age,income,categorical,2,0.0,correlation,0.232950,0.272415
2,adult_income,workclass,income,categorical,2,0.0,cramers_v,0.166965,NaN
3,adult_income,fnlwgt,income,categorical,2,0.0,correlation,-0.012919,-0.013742
4,adult_income,education,income,categorical,2,0.0,cramers_v,0.369462,NaN
5,adult_income,education_num,income,categorical,2,0.0,correlation,0.333750,0.327763
6,adult_income,marital_status,income,categorical,2,0.0,cramers_v,0.444731,NaN
7,adult_income,occupation,income,categorical,2,0.0,cramers_v,0.349804,NaN
8,adult_income,relationship,income,categorical,2,0.0,cramers_v,0.452006,NaN
9,adult_income,race,income,categorical,2,0.0,cramers_v,0.102342,NaN


31

In [21]:
# ============================================================
# NOTEBOOK 04.10 — FEATURE-LEVEL PROFILE ASSEMBLY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.10")
print("ASSEMBLING FEATURE-LEVEL PROFILES")
print("=" * 100)


FEATURE_PROFILE_DF = pd.concat(
    [
        NUMERICAL_PROFILE_DF[
            [
                "dataset_id",
                "feature",
                "dtype",
                "observed_count",
                "missing_count",
                "missing_rate",
                "unique_count",
                "mean",
                "std",
                "variance",
                "min",
                "q1",
                "median",
                "q3",
                "max",
                "skewness",
                "outlier_count",
                "outlier_rate"
            ]
        ],

        CATEGORICAL_PROFILE_DF[
            [
                "dataset_id",
                "feature",
                "dtype",
                "observed_count",
                "missing_count",
                "missing_rate",
                "unique_count",
                "top_category_frequency",
                "entropy",
                "top_categories"
            ]
        ].assign(
            mean=np.nan,
            std=np.nan,
            variance=np.nan,
            min=np.nan,
            q1=np.nan,
            median=np.nan,
            q3=np.nan,
            max=np.nan,
            skewness=np.nan,
            outlier_count=np.nan,
            outlier_rate=np.nan
        )
    ],
    ignore_index=True,
    sort=False
)


FEATURE_PROFILE_DF = (
    FEATURE_PROFILE_DF
    .merge(
        MISSINGNESS_PROFILE_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left",
        suffixes=(
            "",
            "_missingness"
        )
    )
    .merge(
        TASK_RELEVANCE_DF[
            [
                "dataset_id",
                "feature",
                "target_relevance_type",
                "target_relevance_primary",
                "target_relevance_secondary"
            ]
        ],
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
)


# Add MI

if not MUTUAL_INFORMATION_DF.empty:

    FEATURE_PROFILE_DF = (
        FEATURE_PROFILE_DF
        .merge(
            MUTUAL_INFORMATION_DF[
                [
                    "dataset_id",
                    "feature",
                    "mutual_information"
                ]
            ],
            on=[
                "dataset_id",
                "feature"
            ],
            how="left"
        )
    )

else:

    FEATURE_PROFILE_DF[
        "mutual_information"
    ] = np.nan


print(
    f"Feature profiles assembled: "
    f"{len(FEATURE_PROFILE_DF):,}"
)

display(
    FEATURE_PROFILE_DF.head(20)
)

AIR-LLM — NOTEBOOK 04.10
ASSEMBLING FEATURE-LEVEL PROFILES
Feature profiles assembled: 80


,dataset_id,feature,dtype,observed_count,missing_count,missing_rate,unique_count,mean,std,variance,...,missing_percentage,missing_rate_missingness,missing_indicator_mean,mean_row_missing_features_when_missing,mean_row_missing_features_when_observed,missingness_burden,target_relevance_type,target_relevance_primary,target_relevance_secondary,mutual_information
0,adult_income,__air_llm_row_id,int64,22792,0,0.0,22792,16220.811995,9376.973867,8.792764e+07,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.003942,0.003909,0.002678
1,adult_income,age,int64,22792,0,0.0,71,38.583538,13.631512,1.858181e+02,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.232950,0.272415,0.057178
2,adult_income,fnlwgt,int64,22792,0,0.0,16719,190173.628993,105865.625771,1.120753e+10,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,-0.012919,-0.013742,0.013443
3,adult_income,education_num,int64,22792,0,0.0,16,10.074675,2.589412,6.705055e+00,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.333750,0.327763,0.071534
4,adult_income,capital_gain,int64,22792,0,0.0,113,1064.314847,7287.642808,5.310974e+07,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.224387,0.285296,0.083629
5,adult_income,capital_loss,int64,22792,0,0.0,86,86.372762,401.452577,1.611642e+05,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.142852,0.133926,0.036198
6,adult_income,hours_per_week,int64,22792,0,0.0,93,40.469112,12.417125,1.541850e+02,...,0.0,0.0,0.0,0.0,0.130002,0.0,correlation,0.226586,0.264465,0.035510
7,bank_marketing,__air_llm_row_id,int64,31647,0,0.0,31647,22602.388062,13044.705146,1.701643e+08,...,0.0,0.0,0.0,0.0,0.000000,0.0,correlation,0.293029,0.293148,0.085469
8,bank_marketing,age,int64,31647,0,0.0,77,40.883022,10.621809,1.128228e+02,...,0.0,0.0,0.0,0.0,0.000000,0.0,correlation,0.019217,-0.014905,0.014928
9,bank_marketing,balance,int64,31647,0,0.0,6255,1363.589535,3070.235315,9.426345e+06,...,0.0,0.0,0.0,0.0,0.000000,0.0,correlation,0.056386,0.102918,0.009002


In [22]:
# ============================================================
# NOTEBOOK 04.11 — DATASET-LEVEL PROFILE ASSEMBLY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.11")
print("ASSEMBLING DATASET-LEVEL PROFILES")
print("=" * 100)


DATASET_PROFILE_DF = (
    DATASET_PROFILE_DF
    .copy()
)


# Add missingness summary

missing_summary = (
    FEATURE_PROFILE_DF
    .groupby(
        "dataset_id",
        as_index=False
    )
    .agg(

        mean_feature_missing_rate=(
            "missing_rate",
            "mean"
        ),

        max_feature_missing_rate=(
            "missing_rate",
            "max"
        ),

        features_with_missingness=(
            "missing_rate",
            lambda x: int(
                (x > 0).sum()
            )
        ),

        features_high_missingness=(
            "missing_rate",
            lambda x: int(
                (x >= 0.30).sum()
            )
        )
    )
)


DATASET_PROFILE_DF = (
    DATASET_PROFILE_DF
    .merge(
        missing_summary,
        on="dataset_id",
        how="left"
    )
)


# Add task profile

DATASET_PROFILE_DF[
    "task_type"
] = np.where(
    DATASET_PROFILE_DF[
        "target_unique_values"
    ] <= 20,
    "classification_or_low_cardinality",
    "high_cardinality_or_regression"
)


print(
    "Dataset-level profiles assembled."
)

display(
    DATASET_PROFILE_DF
)

AIR-LLM — NOTEBOOK 04.11
ASSEMBLING DATASET-LEVEL PROFILES
Dataset-level profiles assembled.


,dataset_id,row_count,column_count,feature_count,numeric_feature_count,categorical_feature_count,numeric_ratio,categorical_ratio,total_cells,missing_cells,...,duplicate_ratio,target,target_unique_values,target_missing_count,target_missing_rate,mean_feature_missing_rate,max_feature_missing_rate,features_with_missingness,features_high_missingness,task_type
0,adult_income,22792,16,15,7,8,0.466667,0.533333,341880,2963,...,0.0,income,2,0,0.0,0.008667,0.056204,3,0,classification_or_low_cardinality
1,bank_marketing,31647,18,17,8,9,0.470588,0.529412,537999,0,...,0.0,y,2,0,0.0,0.000000,0.000000,0,0,classification_or_low_cardinality
2,diabetes_130us,71236,49,48,12,36,0.250000,0.750000,3419328,261834,...,0.0,readmitted,3,0,0.0,0.076575,0.968415,9,5,classification_or_low_cardinality


In [23]:
# ============================================================
# NOTEBOOK 04.12 — PROFILE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.12")
print("PROFILE VALIDATION")
print("=" * 100)


VALIDATION_ROWS = []


for dataset_id in DATASETS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_REGISTRY[dataset_id]

    profile = FEATURE_PROFILE_DF[
        FEATURE_PROFILE_DF[
            "dataset_id"
        ] == dataset_id
    ]

    expected_features = [
        c for c in df.columns
        if c != target
    ]

    profiled_features = (
        profile["feature"]
        .tolist()
    )

    feature_coverage = (
        set(expected_features)
        == set(profiled_features)
    )

    missing_rate_valid = bool(
        (
            profile["missing_rate"]
            >= 0
        ).all()
        and
        (
            profile["missing_rate"]
            <= 1
        ).all()
    )

    target_excluded = (
        target not in profiled_features
    )

    dataset_profile_exists = (
        dataset_id
        in set(
            DATASET_PROFILE_DF[
                "dataset_id"
            ]
        )
    )

    VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "expected_feature_count":
            len(expected_features),

        "profiled_feature_count":
            len(profiled_features),

        "feature_coverage":
            feature_coverage,

        "missing_rate_valid":
            missing_rate_valid,

        "target_excluded":
            target_excluded,

        "dataset_profile_exists":
            dataset_profile_exists,

        "status":
            (
                "PASS"
                if all([
                    feature_coverage,
                    missing_rate_valid,
                    target_excluded,
                    dataset_profile_exists
                ])
                else "FAIL"
            )
    })


PROFILE_VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)

display(
    PROFILE_VALIDATION_DF
)


if not (
    PROFILE_VALIDATION_DF[
        "status"
    ] == "PASS"
).all():

    raise RuntimeError(
        "Notebook 04 profile validation failed."
    )


print(
    "\nALL PROFILE VALIDATIONS PASSED"
)

AIR-LLM — NOTEBOOK 04.12
PROFILE VALIDATION


,dataset_id,expected_feature_count,profiled_feature_count,feature_coverage,missing_rate_valid,target_excluded,dataset_profile_exists,status
0,adult_income,15,15,True,True,True,True,PASS
1,bank_marketing,17,17,True,True,True,True,PASS
2,diabetes_130us,48,48,True,True,True,True,PASS



ALL PROFILE VALIDATIONS PASSED


In [25]:
# ============================================================
# NOTEBOOK 04.13 — PROFILE PERSISTENCE
# ============================================================

import gc
from pathlib import Path

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.13")
print("PROFILE PERSISTENCE")
print("=" * 100)


# ============================================================
# 1. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "DATASET_PROFILE_DF",
    "FEATURE_PROFILE_DF",
    "MISSINGNESS_PROFILE_DF",
    "NUMERIC_DEPENDENCY_DF",
    "CATEGORICAL_ASSOCIATION_DF",
    "MUTUAL_INFORMATION_DF",
    "TASK_RELEVANCE_DF",
    "PROFILE_VALIDATION_DF"
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:

    raise NameError(
        "The following required profile objects are missing:\n"
        + "\n".join(
            f"  - {obj}"
            for obj in missing_objects
        )
    )


# ============================================================
# 2. REQUIRED DIRECTORY VALIDATION
# ============================================================

required_directories = [
    "DATASET_PROFILE_DIR",
    "FEATURE_PROFILE_DIR",
    "DEPENDENCY_DIR"
]

missing_directories = [
    directory
    for directory in required_directories
    if directory not in globals()
]

if missing_directories:

    raise NameError(
        "The following required profile directories are missing:\n"
        + "\n".join(
            f"  - {directory}"
            for directory in missing_directories
        )
    )


# ============================================================
# 3. RESOLVE METADATA DIRECTORY
# ============================================================

# METADATA_DIR is required for profile validation output.
# If it was not created earlier, derive it from the existing
# profile directory structure when possible.

if "METADATA_DIR" not in globals():

    # Prefer the parent project directory of FEATURE_PROFILE_DIR
    # when the directory structure follows the AIR-LLM layout.

    feature_profile_parent = Path(
        FEATURE_PROFILE_DIR
    ).parent

    candidate_metadata_dir = (
        feature_profile_parent
        / "metadata"
    )

    METADATA_DIR = candidate_metadata_dir


# Ensure the directory exists.

METADATA_DIR = Path(
    METADATA_DIR
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Convert other directories to Path objects.

DATASET_PROFILE_DIR = Path(
    DATASET_PROFILE_DIR
)

FEATURE_PROFILE_DIR = Path(
    FEATURE_PROFILE_DIR
)

DEPENDENCY_DIR = Path(
    DEPENDENCY_DIR
)


# Ensure required directories exist.

DATASET_PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEPENDENCY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. DATASET PROFILE
# ============================================================

DATASET_PROFILE_PATH = (
    DATASET_PROFILE_DIR
    / "dataset_profile.csv"
)

DATASET_PROFILE_DF.to_csv(
    DATASET_PROFILE_PATH,
    index=False
)


# ============================================================
# 5. FEATURE PROFILE
# ============================================================

FEATURE_PROFILE_PATH = (
    FEATURE_PROFILE_DIR
    / "feature_profile.csv"
)

FEATURE_PROFILE_DF.to_csv(
    FEATURE_PROFILE_PATH,
    index=False
)


# ============================================================
# 6. MISSINGNESS PROFILE
# ============================================================

MISSINGNESS_PROFILE_PATH = (
    FEATURE_PROFILE_DIR
    / "missingness_profile.csv"
)

MISSINGNESS_PROFILE_DF.to_csv(
    MISSINGNESS_PROFILE_PATH,
    index=False
)


# ============================================================
# 7. NUMERICAL DEPENDENCIES
# ============================================================

NUMERIC_DEPENDENCY_PATH = (
    DEPENDENCY_DIR
    / "numeric_dependencies.csv"
)

NUMERIC_DEPENDENCY_DF.to_csv(
    NUMERIC_DEPENDENCY_PATH,
    index=False
)


# ============================================================
# 8. CATEGORICAL ASSOCIATIONS
# ============================================================

CATEGORICAL_ASSOCIATION_PATH = (
    DEPENDENCY_DIR
    / "categorical_associations.csv"
)

CATEGORICAL_ASSOCIATION_DF.to_csv(
    CATEGORICAL_ASSOCIATION_PATH,
    index=False
)


# ============================================================
# 9. MUTUAL INFORMATION
# ============================================================

MI_PATH = (
    DEPENDENCY_DIR
    / "mutual_information.csv"
)

MUTUAL_INFORMATION_DF.to_csv(
    MI_PATH,
    index=False
)


# ============================================================
# 10. TASK RELEVANCE
# ============================================================

TASK_RELEVANCE_PATH = (
    FEATURE_PROFILE_DIR
    / "task_relevance.csv"
)

TASK_RELEVANCE_DF.to_csv(
    TASK_RELEVANCE_PATH,
    index=False
)


# ============================================================
# 11. PROFILE VALIDATION
# ============================================================

VALIDATION_PATH = (
    METADATA_DIR
    / "profile_validation.csv"
)

PROFILE_VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)


# ============================================================
# 12. VERIFY FILE CREATION
# ============================================================

PROFILE_PATHS = [

    DATASET_PROFILE_PATH,

    FEATURE_PROFILE_PATH,

    MISSINGNESS_PROFILE_PATH,

    NUMERIC_DEPENDENCY_PATH,

    CATEGORICAL_ASSOCIATION_PATH,

    MI_PATH,

    TASK_RELEVANCE_PATH,

    VALIDATION_PATH

]


print("\nPROFILE FILES SAVED")
print("-" * 100)


save_failures = []

for path in PROFILE_PATHS:

    path = Path(path)

    if path.exists():

        size_kb = (
            path.stat().st_size / 1024
        )

        print(
            f"  ✓ {path} "
            f"({size_kb:.2f} KB)"
        )

    else:

        save_failures.append(
            str(path)
        )

        print(
            f"  ✗ FAILED: {path}"
        )


# ============================================================
# 13. PERSISTENCE VALIDATION
# ============================================================

print("\n" + "=" * 100)
print("PROFILE PERSISTENCE VALIDATION")
print("=" * 100)

if save_failures:

    print(
        f"FAILED FILES: {len(save_failures)}"
    )

    for failure in save_failures:

        print(
            f"  - {failure}"
        )

    raise RuntimeError(
        "One or more profile files were not saved successfully."
    )

else:

    print(
        f"All {len(PROFILE_PATHS)} profile files "
        f"saved successfully."
    )


# ============================================================
# 14. RESULT SUMMARY
# ============================================================

print("\nPROFILE DATASET SIZES")
print("-" * 100)

print(
    f"Dataset profile           : "
    f"{len(DATASET_PROFILE_DF):,} rows"
)

print(
    f"Feature profile           : "
    f"{len(FEATURE_PROFILE_DF):,} rows"
)

print(
    f"Missingness profile       : "
    f"{len(MISSINGNESS_PROFILE_DF):,} rows"
)

print(
    f"Numeric dependencies      : "
    f"{len(NUMERIC_DEPENDENCY_DF):,} rows"
)

print(
    f"Categorical associations  : "
    f"{len(CATEGORICAL_ASSOCIATION_DF):,} rows"
)

print(
    f"Mutual information       : "
    f"{len(MUTUAL_INFORMATION_DF):,} rows"
)

print(
    f"Task relevance            : "
    f"{len(TASK_RELEVANCE_DF):,} rows"
)

print(
    f"Profile validation        : "
    f"{len(PROFILE_VALIDATION_DF):,} rows"
)


# ============================================================
# 15. MEMORY CLEANUP
# ============================================================

gc.collect()

print("\n" + "=" * 100)
print("NOTEBOOK 04.13 — PROFILE PERSISTENCE COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 04.13
PROFILE PERSISTENCE

PROFILE FILES SAVED
----------------------------------------------------------------------------------------------------
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/dataset/dataset_profile.csv (0.92 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/feature/feature_profile.csv (20.99 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/feature/missingness_profile.csv (6.43 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/dependency/numeric_dependencies.csv (10.56 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/dependency/categorical_associations.csv (29.73 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/dependency/mutual_information.csv (6.40 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/feature/task_relevance.csv (7.11 KB)
  ✓ /content/drive/MyDrive/AIR_LLM_Research/d

In [26]:
# ============================================================
# NOTEBOOK 04.14 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.14")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)


REQUIRED_OUTPUTS = [

    DATASET_PROFILE_PATH,

    FEATURE_PROFILE_PATH,

    MISSINGNESS_PROFILE_PATH,

    NUMERIC_DEPENDENCY_PATH,

    CATEGORICAL_ASSOCIATION_PATH,

    MI_PATH,

    TASK_RELEVANCE_PATH,

    VALIDATION_PATH
]


PERSISTENCE_ROWS = []


for path in REQUIRED_OUTPUTS:

    exists = path.is_file()

    size = (
        path.stat().st_size
        if exists
        else 0
    )

    PERSISTENCE_ROWS.append({

        "file":
            path.name,

        "exists":
            exists,

        "size_bytes":
            size,

        "valid":
            bool(
                exists
                and size > 0
            )
    })


PERSISTENCE_DF = pd.DataFrame(
    PERSISTENCE_ROWS
)

display(
    PERSISTENCE_DF
)


if not PERSISTENCE_DF[
    "valid"
].all():

    raise RuntimeError(
        "One or more Notebook 04 outputs "
        "were not persisted correctly."
    )


print(
    "\nDrive persistence validation: PASSED"
)

AIR-LLM — NOTEBOOK 04.14
DRIVE PERSISTENCE VALIDATION


,file,exists,size_bytes,valid
0,dataset_profile.csv,True,945,True
1,feature_profile.csv,True,21498,True
2,missingness_profile.csv,True,6584,True
3,numeric_dependencies.csv,True,10809,True
4,categorical_associations.csv,True,30448,True
5,mutual_information.csv,True,6557,True
6,task_relevance.csv,True,7278,True
7,profile_validation.csv,True,275,True



Drive persistence validation: PASSED


In [28]:
# ============================================================
# NOTEBOOK 04.15 — MANIFEST
# ============================================================

import gc
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04.15")
print("NOTEBOOK 04 MANIFEST")
print("=" * 100)


# ============================================================
# 1. VALIDATE REQUIRED GLOBAL OBJECTS
# ============================================================

required_objects = [
    "PROJECT_ROOT",
    "MASTER_SEED",
    "DATASETS",
    "TARGET_REGISTRY",
    "PROFILE_SAMPLE_SIZE",
    "DATASET_PROFILE_DF",
    "FEATURE_PROFILE_DF",
    "NUMERIC_DEPENDENCY_DF",
    "CATEGORICAL_ASSOCIATION_DF",
    "MUTUAL_INFORMATION_DF"
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:

    raise NameError(
        "Required AIR-LLM objects are missing:\n"
        + "\n".join(
            f"  - {obj}"
            for obj in missing_objects
        )
    )


# ============================================================
# 2. NORMALIZE PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path(
    PROJECT_ROOT
).resolve()


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist: "
        f"{PROJECT_ROOT}"
    )


# ============================================================
# 3. DEFINE MANIFEST DIRECTORY
# ============================================================

# Prefer an existing metadata directory.
# Otherwise create a dedicated manifests directory.

if "METADATA_DIR" in globals():

    MANIFEST_DIR = Path(
        METADATA_DIR
    )

else:

    MANIFEST_DIR = (
        PROJECT_ROOT
        / "metadata"
    )


MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. DEFINE MANIFEST PATH
# ============================================================

MANIFEST_PATH = (
    MANIFEST_DIR
    / "notebook_04_manifest.json"
)


# ============================================================
# 5. DEFINE SHA-256 FUNCTION
# ============================================================

def sha256_file(path):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


# ============================================================
# 6. BUILD REQUIRED OUTPUT LIST
# ============================================================

# Use the previously defined REQUIRED_OUTPUTS when available.
# Otherwise construct the expected Notebook 04 outputs.

if "REQUIRED_OUTPUTS" not in globals():

    REQUIRED_OUTPUTS = [

        Path(DATASET_PROFILE_DIR)
        / "dataset_profile.csv",

        Path(FEATURE_PROFILE_DIR)
        / "feature_profile.csv",

        Path(FEATURE_PROFILE_DIR)
        / "missingness_profile.csv",

        Path(DEPENDENCY_DIR)
        / "numeric_dependencies.csv",

        Path(DEPENDENCY_DIR)
        / "categorical_associations.csv",

        Path(DEPENDENCY_DIR)
        / "mutual_information.csv",

        Path(FEATURE_PROFILE_DIR)
        / "task_relevance.csv",

        Path(METADATA_DIR)
        / "profile_validation.csv"

    ]

else:

    REQUIRED_OUTPUTS = [
        Path(path)
        for path in REQUIRED_OUTPUTS
    ]


# ============================================================
# 7. BUILD OUTPUT MANIFEST
# ============================================================

OUTPUT_MANIFEST = {}

missing_outputs = []

for path in REQUIRED_OUTPUTS:

    path = Path(path)

    # --------------------------------------------------------
    # Convert absolute paths safely
    # --------------------------------------------------------

    if path.is_absolute():

        absolute_path = path

    else:

        absolute_path = (
            PROJECT_ROOT
            / path
        )

    absolute_path = (
        absolute_path
        .resolve()
    )


    # --------------------------------------------------------
    # Relative path for manifest
    # --------------------------------------------------------

    try:

        relative_path = str(
            absolute_path.relative_to(
                PROJECT_ROOT
            )
        )

    except ValueError:

        relative_path = str(
            absolute_path
        )


    # --------------------------------------------------------
    # Existing file
    # --------------------------------------------------------

    if absolute_path.is_file():

        OUTPUT_MANIFEST[
            relative_path
        ] = {

            "exists":
                True,

            "size_bytes":
                int(
                    absolute_path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    absolute_path
                )

        }


    # --------------------------------------------------------
    # Missing file
    # --------------------------------------------------------

    else:

        OUTPUT_MANIFEST[
            relative_path
        ] = {

            "exists":
                False,

            "size_bytes":
                0,

            "sha256":
                None

        }

        missing_outputs.append(
            relative_path
        )


# ============================================================
# 8. BUILD NOTEBOOK MANIFEST
# ============================================================

MANIFEST = {

    "project":
        "AIR-LLM",

    "notebook":
        "04_Dataset_and_Feature_Profiling",

    "version":
        "1.0",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "master_seed":
        int(
            MASTER_SEED
        ),

    "datasets":
        list(
            DATASETS
        ),

    "target_registry":
        dict(
            TARGET_REGISTRY
        ),

    "profile_sample_size":
        int(
            PROFILE_SAMPLE_SIZE
        ),

    "dataset_profile_rows":
        int(
            len(
                DATASET_PROFILE_DF
            )
        ),

    "feature_profile_rows":
        int(
            len(
                FEATURE_PROFILE_DF
            )
        ),

    "missingness_profile_rows":
        (
            int(
                len(
                    MISSINGNESS_PROFILE_DF
                )
            )
            if "MISSINGNESS_PROFILE_DF" in globals()
            else None
        ),

    "numeric_dependency_rows":
        int(
            len(
                NUMERIC_DEPENDENCY_DF
            )
        ),

    "categorical_association_rows":
        int(
            len(
                CATEGORICAL_ASSOCIATION_DF
            )
        ),

    "mutual_information_rows":
        int(
            len(
                MUTUAL_INFORMATION_DF
            )
        ),

    "task_relevance_rows":
        (
            int(
                len(
                    TASK_RELEVANCE_DF
                )
            )
            if "TASK_RELEVANCE_DF" in globals()
            else None
        ),

    "profile_validation_rows":
        (
            int(
                len(
                    PROFILE_VALIDATION_DF
                )
            )
            if "PROFILE_VALIDATION_DF" in globals()
            else None
        ),

    "output_count":
        int(
            len(
                OUTPUT_MANIFEST
            )
        ),

    "existing_output_count":
        int(
            sum(
                item["exists"]
                for item in OUTPUT_MANIFEST.values()
            )
        ),

    "missing_output_count":
        int(
            len(
                missing_outputs
            )
        ),

    "outputs":
        OUTPUT_MANIFEST
}


# ============================================================
# 9. SAVE MANIFEST
# ============================================================

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 10. VERIFY MANIFEST
# ============================================================

if not MANIFEST_PATH.is_file():

    raise RuntimeError(
        "Manifest file was not created successfully."
    )


manifest_size = (
    MANIFEST_PATH.stat().st_size
)


# ============================================================
# 11. REPORT
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 04 MANIFEST SUMMARY")
print("=" * 100)

print(
    f"Manifest saved        : {MANIFEST_PATH}"
)

print(
    f"Manifest size         : "
    f"{manifest_size:,} bytes"
)

print(
    f"Output artifacts      : "
    f"{len(OUTPUT_MANIFEST):,}"
)

print(
    f"Existing artifacts    : "
    f"{MANIFEST['existing_output_count']:,}"
)

print(
    f"Missing artifacts     : "
    f"{MANIFEST['missing_output_count']:,}"
)


# ============================================================
# 12. OUTPUT STATUS
# ============================================================

print("\nOUTPUT STATUS")
print("-" * 100)

for relative_path, metadata in OUTPUT_MANIFEST.items():

    if metadata["exists"]:

        print(
            f"  ✓ {relative_path}"
        )

    else:

        print(
            f"  ✗ MISSING: {relative_path}"
        )


# ============================================================
# 13. FINAL VALIDATION
# ============================================================

if missing_outputs:

    print("\n" + "=" * 100)
    print("WARNING — INCOMPLETE NOTEBOOK 04 OUTPUTS")
    print("=" * 100)

    print(
        "The manifest was created, but the following "
        "expected outputs are missing:"
    )

    for output in missing_outputs:

        print(
            f"  - {output}"
        )

    print(
        "\nDo not treat Notebook 04 as fully complete "
        "until these outputs are generated."
    )

else:

    print("\n" + "=" * 100)
    print("NOTEBOOK 04 — ALL OUTPUTS VERIFIED")
    print("=" * 100)

    print(
        "All expected profiling artifacts exist "
        "and have been hashed successfully."
    )


# ============================================================
# 14. MEMORY CLEANUP
# ============================================================

gc.collect()

print("\n" + "=" * 100)
print("NOTEBOOK 04.15 — MANIFEST COMPLETE")
print("=" * 100)

AIR-LLM — NOTEBOOK 04.15
NOTEBOOK 04 MANIFEST

NOTEBOOK 04 MANIFEST SUMMARY
Manifest saved        : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_04/profiles/metadata/notebook_04_manifest.json
Manifest size         : 2,407 bytes
Output artifacts      : 8
Existing artifacts    : 8
Missing artifacts     : 0

OUTPUT STATUS
----------------------------------------------------------------------------------------------------
  ✓ data/notebook_04/profiles/dataset/dataset_profile.csv
  ✓ data/notebook_04/profiles/feature/feature_profile.csv
  ✓ data/notebook_04/profiles/feature/missingness_profile.csv
  ✓ data/notebook_04/profiles/dependency/numeric_dependencies.csv
  ✓ data/notebook_04/profiles/dependency/categorical_associations.csv
  ✓ data/notebook_04/profiles/dependency/mutual_information.csv
  ✓ data/notebook_04/profiles/feature/task_relevance.csv
  ✓ data/notebook_04/profiles/metadata/profile_validation.csv

NOTEBOOK 04 — ALL OUTPUTS VERIFIED
All expected profiling artifacts exi

In [29]:
# ============================================================
# NOTEBOOK 04.16 — FINAL NOTEBOOK STATUS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 04 COMPLETE")
print("=" * 100)


print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASETS:

    row = DATASET_PROFILE_DF[
        DATASET_PROFILE_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    print(
        f"{dataset_id:20s} | "
        f"Rows: {int(row['row_count']):8,d} | "
        f"Features: {int(row['feature_count']):3d} | "
        f"Missing: {row['overall_missing_rate']:.4f}"
    )


print("\nPROFILES")
print("-" * 100)

print(
    f"Dataset profiles       : "
    f"{len(DATASET_PROFILE_DF)}"
)

print(
    f"Feature profiles       : "
    f"{len(FEATURE_PROFILE_DF)}"
)

print(
    f"Missingness profiles   : "
    f"{len(MISSINGNESS_PROFILE_DF)}"
)

print(
    f"Numeric dependencies   : "
    f"{len(NUMERIC_DEPENDENCY_DF):,}"
)

print(
    f"Categorical associations: "
    f"{len(CATEGORICAL_ASSOCIATION_DF):,}"
)

print(
    f"Mutual-information rows: "
    f"{len(MUTUAL_INFORMATION_DF):,}"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Feature coverage       : PASSED"
)

print(
    "Missingness validation : PASSED"
)

print(
    "Target exclusion       : PASSED"
)

print(
    "Profile persistence     : PASSED"
)

print(
    "Drive persistence      : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    f"Dataset profile :\n{DATASET_PROFILE_PATH}"
)

print(
    f"Feature profile :\n{FEATURE_PROFILE_PATH}"
)

print(
    f"Missingness     :\n{MISSINGNESS_PROFILE_PATH}"
)

print(
    f"Dependencies    :\n{DEPENDENCY_DIR}"
)

print(
    f"Manifest        :\n{MANIFEST_PATH}"
)


print("\n" + "=" * 100)
print("ALL NOTEBOOK 04 VALIDATIONS PASSED")
print("AIR-LLM DATASET AND FEATURE PROFILES ARE READY")
print("NOTEBOOK 05 — LLM RECOMMENDATION ENGINE MAY BEGIN")
print("=" * 100)

AIR-LLM — NOTEBOOK 04 COMPLETE

DATASETS
----------------------------------------------------------------------------------------------------
adult_income         | Rows:   22,792 | Features:  15 | Missing: 0.0087
bank_marketing       | Rows:   31,647 | Features:  17 | Missing: 0.0000
diabetes_130us       | Rows:   71,236 | Features:  48 | Missing: 0.0766

PROFILES
----------------------------------------------------------------------------------------------------
Dataset profiles       : 3
Feature profiles       : 80
Missingness profiles   : 80
Numeric dependencies   : 115
Categorical associations: 414
Mutual-information rows: 80

VALIDATION
----------------------------------------------------------------------------------------------------
Feature coverage       : PASSED
Missingness validation : PASSED
Target exclusion       : PASSED
Profile persistence     : PASSED
Drive persistence      : PASSED

OUTPUTS
------------------------------------------------------------------------------